# N27 — Race Situation Agent

This notebook builds the **Race Situation Agent**, the third sub-agent in the F1
multi-agent system. It answers two questions per lap:

- **Can we overtake?** — probability that driver X passes the car directly ahead
  within the next few laps, given the current gap, pace delta, and tyre context.
- **Is a Safety Car coming?** — probability of a SC deployment within the next
  3 laps, given the current race chaos level.

Both questions are answered by LightGBM classifiers trained in N12 and N14,
wrapped in LangChain `@tool` functions and orchestrated by a LangGraph ReAct agent.

Models loaded from `data/models/`:

| File | Source | Contents |
|---|---|---|
| `overtake_probability/lgbm_overtake_v1.pkl` | N12 | LightGBM overtake classifier |
| `overtake_probability/calibrator.pkl` | N12 | Platt calibrator (val 2024) |
| `overtake_probability/model_config.json` | N12 | Features, threshold, CAT_FEATURES |
| `safety_car_probability/lgbm_sc_v1.pkl` | N14 | LightGBM SC classifier |
| `safety_car_probability/calibrator_sc_v1.pkl` | N14 | Platt calibrator |
| `safety_car_probability/feature_list_v1.json` | N14 | Feature list (ordered) |

Output consumed by N31 Orchestrator to condition the pit strategy decision in N28.


---

## Step 0 — Setup & model loading

Imports, repo root, FastF1 cache. `RaceSituationConfig` loads both model pairs
(overtake + SC) and the circuit cluster map. All models serialised with `joblib`
(required on Windows paths containing non-ASCII characters).


In [ ]:
import json, sys, re
from dataclasses import dataclass, field
from pathlib import Path

import fastf1
import joblib
import numpy as np
import pandas as pd

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# Module-level globals — populated by setup_smoke_test_session() before tool invocation
LAPS: pd.DataFrame = pd.DataFrame()
SESSION_META: dict = {}

# Absolute tyre compound map (SOFT/MEDIUM/HARD → C1-C6 per circuit/year)
_tire_compounds_path = repo_root / "data" / "tire_compounds_by_race.json"
TIRE_COMPOUNDS: dict = json.loads(_tire_compounds_path.read_text(encoding="utf-8")) if _tire_compounds_path.exists() else {}

In [2]:
@dataclass
class RaceSituationConfig:
    """Runtime configuration for the Race Situation Agent.

    Loads both LightGBM model pairs (overtake from N12, SC from N14) plus their
    Platt calibrators and feature lists. Models are loaded with joblib because the
    repo path contains non-ASCII characters that break LightGBM's native save_model
    on Windows.

    overtake_threshold and sc_threshold are the classification thresholds derived
    from the Optuna/F2-score tuning in N12 and N14 respectively. threat_level
    boundaries (high_overtake, high_sc, medium_overtake, medium_sc) are agent-level
    decision rules that map raw probabilities to LOW/MEDIUM/HIGH for N31.
    """

    model_name: str = "local-model"

    # Threat-level boundaries
    high_overtake: float   = 0.80   # N12 optimal threshold — above this → HIGH
    medium_overtake: float = 0.40
    high_sc: float         = 0.30   # above this → HIGH regardless of overtake
    medium_sc: float       = 0.15

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent

        fastf1.Cache.enable_cache(str(root / "data" / "cache" / "fastf1"))

        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        # --- Overtake model (N12) ---
        _ov_dir = root / "data" / "models" / "overtake_probability"
        self.overtake_model      = joblib.load(_ov_dir / "lgbm_overtake_v1.pkl")
        self.overtake_calibrator = joblib.load(_ov_dir / "calibrator.pkl")
        with open(_ov_dir / "model_config.json") as f:
            ov_cfg = json.load(f)
        self.overtake_features: list[str]     = ov_cfg["features"]
        self.overtake_cat_features: list[str] = ov_cfg["categorical_features"]
        self.overtake_threshold: float        = ov_cfg["optimal_threshold"]

        # --- SC model (N14) ---
        _sc_dir = root / "data" / "models" / "safety_car_probability"
        self.sc_model      = joblib.load(_sc_dir / "lgbm_sc_v1.pkl")
        self.sc_calibrator = joblib.load(_sc_dir / "calibrator_sc_v1.pkl")
        with open(_sc_dir / "feature_list_v1.json") as f:
            _sc_cfg = json.load(f)
        self.sc_features: list[str] = _sc_cfg["features"]
        self.sc_threshold: float    = _sc_cfg["best_threshold"]

        # --- Circuit cluster map (N03) — keyed by GP_Name ---
        _cluster_df = pd.read_parquet(
            root / "data" / "processed" / "circuit_clustering" / "circuit_clusters_k4.parquet"
        )
        self.circuit_cluster_map: dict = dict(
            zip(_cluster_df["GP_Name"], _cluster_df["Cluster"].astype(int))
        )

        # --- Circuit SC base rates (N13) — keyed by event_name ---
        _sc_rates_path = root / "data" / "processed" / "sc_labeled" / "sc_labeled_2023_2025.parquet"
        _sc_df = pd.read_parquet(_sc_rates_path, columns=["event_name", "circuit_sc_rate"])
        self.circuit_sc_rate_map: dict = (
            _sc_df.drop_duplicates("event_name")
            .set_index("event_name")["circuit_sc_rate"]
            .to_dict()
        )

In [3]:
CFG = RaceSituationConfig()

print(f"Overtake model     : {type(CFG.overtake_model).__name__}")
print(f"Overtake features  ({len(CFG.overtake_features)}): {CFG.overtake_features}")
print(f"Overtake cat       : {CFG.overtake_cat_features}")
print(f"Overtake threshold : {CFG.overtake_threshold:.4f}")
print()
print(f"SC model           : {type(CFG.sc_model).__name__}")
print(f"SC features        ({len(CFG.sc_features)}): {CFG.sc_features[:6]} ...")
print(f"SC threshold       : {CFG.sc_threshold:.4f}")
print()
print(f"Circuits in cluster map  : {len(CFG.circuit_cluster_map)}")
print(f"Circuits in SC rate map  : {len(CFG.circuit_sc_rate_map)}")

Overtake model     : LGBMClassifier
Overtake features  (15): ['gap_ahead_s', 'pace_delta_s', 'tyre_life_x', 'tyre_life_y', 'tyre_life_diff', 'speed_trap_delta', 'LapNumber', 'drs_window', 'compound_x', 'compound_y', 'circuit_cluster', 'gap_pace_product', 'drs_ready_gap', 'gap_trend', 'pace_delta_rolling3']
Overtake cat       : ['compound_x', 'compound_y', 'circuit_cluster']
Overtake threshold : 0.7976

SC model           : LGBMClassifier
SC features        (32): ['lap_time_mean_z', 'lap_time_std_z', 'lap_time_min_z', 'lap_time_cv', 'lap_time_trend_5', 'n_drivers'] ...
SC threshold       : 0.2335

Circuits in cluster map  : 25
Circuits in SC rate map  : 23


### Step 0 results

| Model | Type | Features | Threshold |
|---|---|---|---|
| Overtake (N12) | LGBMClassifier | 15 | 0.7976 |
| SC (N14) | LGBMClassifier | 32 | 0.2335 |

- Circuit cluster map: 25 circuits
- SC rate map: 23 circuits


---

## Step 1 — `RaceSituationOutput` dataclass + feature helpers

Defines the agent's output structure and the feature engineering pipeline that
converts lap-by-lap FastF1 data into the exact feature sets expected by the N12
overtake model (15 features) and N14 SC model (32 features).

Feature engineering replicates the N12/N14 training pipelines exactly — same
column names, same rolling computations, same categorical encodings. The pipeline
is split into focused helper functions (one per model) so each step is
independently readable and testable.


`RaceSituationOutput` dataclass — structured agent output with `threat_level`
derived from both `overtake_prob` and `sc_prob_3lap` using the thresholds in `CFG`.


In [4]:
@dataclass
class RaceSituationOutput:
    """Structured output of the Race Situation Agent for one lap snapshot.

    Combines overtaking opportunity assessment (N12) with safety car risk
    prediction (N14) into a single threat level classification that the
    Strategy Orchestrator (N31) uses to condition pit timing and stint
    extension decisions.

    The threat level is derived automatically in __post_init__ so downstream
    agents receive a categorical signal (LOW/MEDIUM/HIGH) without
    re-implementing threshold logic. The raw probabilities are preserved for
    logging and post-race analysis.

    Attributes:
        overtake_prob: Calibrated probability (0-1) that the driver will
                       overtake the car directly ahead within the next few laps.
                       Derived from N12 LightGBM + Platt calibration. Values
                       above CFG.high_overtake (0.80) indicate a strong
                       opportunity — useful for deciding whether to push pace
                       or box early to avoid losing track position.
        sc_prob_3lap: Calibrated probability (0-1) that a Safety Car will be
                      deployed within the next 3 laps. Derived from N14
                      LightGBM + Platt calibration. Values above CFG.high_sc
                      (0.30) flag imminent SC risk — pit now before the SC
                      closes the pit lane or causes a chaotic reshuffling.
        threat_level: Categorical assessment derived from the two probabilities.
                      LOW: clear air, no urgent threats → extend stint freely.
                      MEDIUM: traffic ahead or moderate SC risk → monitor closely.
                      HIGH: strong overtake opportunity OR imminent SC → act now.
        gap_ahead_s: Gap in seconds to the car directly ahead at this lap.
                     Stored for context logging. Gaps < 1.0s indicate DRS range.
        pace_delta_s: Pace advantage or deficit (s/lap) vs the car ahead, computed
                      as a 3-lap rolling mean. Negative values mean we are faster.
                      Stored for context logging and to validate model inputs.
        reasoning: Human-readable synthesis from the LLM agent explaining why
                   this threat level was assigned. Used by N31 for strategy
                   explanation and by post-race reports for interpretability.
    """

    overtake_prob: float
    sc_prob_3lap: float
    threat_level: str = field(init=False)
    gap_ahead_s: float = 0.0
    pace_delta_s: float = 0.0
    reasoning: str = ""

    def __post_init__(self):
        """Derive threat_level from the two probability signals.

        HIGH is triggered by either:
          - overtake_prob >= CFG.high_overtake (strong passing opportunity)
          - sc_prob_3lap >= CFG.high_sc (imminent SC deployment)

        MEDIUM is triggered by either:
          - overtake_prob >= CFG.medium_overtake (rival blocking, consider undercut)
          - sc_prob_3lap >= CFG.medium_sc (elevated SC risk, prepare contingency)

        LOW is the default when both probabilities are below medium thresholds
        (clear air, low chaos, no urgent strategic pressure).
        """
        if self.overtake_prob >= CFG.high_overtake or self.sc_prob_3lap >= CFG.high_sc:
            self.threat_level = "HIGH"
        elif self.overtake_prob >= CFG.medium_overtake or self.sc_prob_3lap >= CFG.medium_sc:
            self.threat_level = "MEDIUM"
        else:
            self.threat_level = "LOW"


In [5]:
# ── Feature engineering constants (from N13/N14 training definitions) ────────

# Tyre-age cliff thresholds — beyond these laps the compound degrades sharply
CLIFF_THRESHOLDS = {"SOFT": 20, "MEDIUM": 35, "HARD": 50}

# Track status ordinal encoding (0=green → 5=SC) — from N13 STATUS_ENC
STATUS_ENC = {"1": 0, "2": 1, "5": 2, "7": 3, "6": 4, "4": 5}

# Severity scale for change-direction features — higher = more dangerous
STATUS_SEVERITY = {"1": 1, "2": 2, "5": 3, "7": 4, "6": 5, "4": 6}

# RCM incident pattern (from N13 build_clean_incident_mask)
_INCIDENT_RE = r"INCIDENT|COLLISION|CONTACT|SPIN|OFF TRACK|STOPPED CAR|DEBRIS|MARSHAL"
_EXCLUDE_RE  = r"TRACK LIMITS|LAP TIME|PENALTY|PIT LANE|FORMATION|GRID|DRS|SAFETY CAR|VIRTUAL"

Smoke test dataclass

In [6]:
def smoke_test_race_situation_output():
    """Verify RaceSituationOutput instantiates and derives threat_level correctly."""
    # HIGH via overtake
    out1 = RaceSituationOutput(overtake_prob=0.85, sc_prob_3lap=0.10, gap_ahead_s=0.9, pace_delta_s=-0.2)
    print(f"Test 1 (HIGH via overtake): {out1.threat_level} | ov={out1.overtake_prob} sc={out1.sc_prob_3lap}")

    # HIGH via SC
    out2 = RaceSituationOutput(overtake_prob=0.20, sc_prob_3lap=0.35, gap_ahead_s=3.5, pace_delta_s=0.1)
    print(f"Test 2 (HIGH via SC):       {out2.threat_level} | ov={out2.overtake_prob} sc={out2.sc_prob_3lap}")

    # MEDIUM via overtake
    out3 = RaceSituationOutput(overtake_prob=0.50, sc_prob_3lap=0.08, gap_ahead_s=1.8, pace_delta_s=-0.05)
    print(f"Test 3 (MEDIUM via ov):     {out3.threat_level} | ov={out3.overtake_prob} sc={out3.sc_prob_3lap}")

    # LOW
    out4 = RaceSituationOutput(overtake_prob=0.15, sc_prob_3lap=0.05, gap_ahead_s=5.0, pace_delta_s=0.0)
    print(f"Test 4 (LOW):               {out4.threat_level} | ov={out4.overtake_prob} sc={out4.sc_prob_3lap}")

smoke_test_race_situation_output()


Test 1 (HIGH via overtake): HIGH | ov=0.85 sc=0.1
Test 2 (HIGH via SC):       HIGH | ov=0.2 sc=0.35
Test 3 (MEDIUM via ov):     MEDIUM | ov=0.5 sc=0.08
Test 4 (LOW):               LOW | ov=0.15 sc=0.05


Feature engineering helpers — split into two focused functions, one per model.
`build_overtake_features` constructs the 15 N12 features from a pair of driver
laps. `build_sc_features` constructs the 32 N14 features from the full field's
recent lap times and race control message context.


In [7]:
def _abs_compound(relative: str, gp_name: str, year: int) -> str:
    """Map SOFT/MEDIUM/HARD → C1-C6 using TIRE_COMPOUNDS; fallback to relative name."""
    return TIRE_COMPOUNDS.get(str(year), {}).get(gp_name, {}).get(relative.upper(), relative)


def build_overtake_features(
    driver_x_lap: pd.Series,
    driver_y_lap: pd.Series,
    laps_recent: pd.DataFrame,
    circuit_cluster: int,
    gp_name: str = "",
    year: int = 2025,
) -> pd.DataFrame:
    """Build the 15 N12 overtake model features from a driver pair at one lap.

    Replicates the N12 training feature pipeline exactly. driver_x is the chasing
    car (the one attempting the overtake), driver_y is the car directly ahead.
    laps_recent must contain at least 3 laps of history for both drivers to
    compute rolling pace trends (pace_delta_rolling3, gap_trend).

    Args:
        driver_x_lap: FastF1 lap Series for the chasing driver at the current lap.
                      Must contain: LapTime, TyreLife, Compound, SpeedST, LapNumber.
                      If the "Time" column (session elapsed time) is present it is
                      used for an accurate on-track gap; otherwise falls back to a
                      lap-time-difference approximation.
        driver_y_lap: FastF1 lap Series for the car directly ahead.
        laps_recent: DataFrame of laps for both drivers over the last 3+ laps with
                     columns Driver, LapNumber, LapTime, Time (optional).
        circuit_cluster: Integer cluster ID (0-3) from CFG.circuit_cluster_map.
        gp_name: GP short name (e.g., 'Sakhir') used to look up absolute compound.
        year: Race year for compound lookup.

    Returns:
        Single-row DataFrame with 15 columns in CFG.overtake_features order.
        compound_x/y are absolute C-strings (C1-C6) cast to pandas category so
        LightGBM applies the same categorical encoding as during N12 training.
    """
    # ── gap_ahead_s: on-track gap via cumulative session elapsed time ─────────
    # FastF1 "Time" = session time at lap-end crossing; if Y is ahead then Y.Time < X.Time
    t_x = driver_x_lap.get("Time")
    t_y = driver_y_lap.get("Time")
    if pd.notna(t_x) and pd.notna(t_y):
        gap_ahead_s = float((t_x - t_y).total_seconds())
    else:
        # Fallback: raw lap-time difference (less accurate but safe)
        gap_ahead_s = float(
            (driver_x_lap["LapTime"] - driver_y_lap["LapTime"]).total_seconds()
        )
    gap_ahead_s = max(0.0, gap_ahead_s)

    # ── Pace delta (per lap, positive = X is slower) ──────────────────────────
    pace_delta_s = float(
        (driver_x_lap["LapTime"] - driver_y_lap["LapTime"]).total_seconds()
    )

    # ── Tyre life ─────────────────────────────────────────────────────────────
    tyre_life_x   = int(driver_x_lap["TyreLife"])
    tyre_life_y   = int(driver_y_lap["TyreLife"])
    tyre_life_diff = tyre_life_x - tyre_life_y

    # ── Speed trap delta, lap number ─────────────────────────────────────────
    speed_trap_delta = float(driver_x_lap.get("SpeedST", 300.0)) - float(driver_y_lap.get("SpeedST", 300.0))
    lap_number       = int(driver_x_lap["LapNumber"])

    # ── DRS ───────────────────────────────────────────────────────────────────
    drs_window    = int(gap_ahead_s < 1.0)
    drs_ready_gap = gap_ahead_s * drs_window   # 0 when not in DRS window

    # ── Compound (absolute via TIRE_COMPOUNDS map) ────────────────────────────
    compound_x = _abs_compound(str(driver_x_lap.get("Compound", "MEDIUM")), gp_name, year)
    compound_y = _abs_compound(str(driver_y_lap.get("Compound", "MEDIUM")), gp_name, year)

    # ── Derived ───────────────────────────────────────────────────────────────
    gap_pace_product = gap_ahead_s * pace_delta_s

    # ── Rolling trends from laps_recent (last 3 laps per driver) ─────────────
    _dx = laps_recent[laps_recent["Driver"] == driver_x_lap["Driver"]].sort_values("LapNumber").tail(3)
    _dy = laps_recent[laps_recent["Driver"] == driver_y_lap["Driver"]].sort_values("LapNumber").tail(3)

    if len(_dx) >= 2 and len(_dy) >= 2:
        # pace_delta_rolling3: 3-lap mean of pace delta (N12 definition)
        _dx_t = _dx["LapTime"].dt.total_seconds().values
        _dy_t = _dy["LapTime"].dt.total_seconds().values
        n_shared = min(len(_dx_t), len(_dy_t))
        pace_delta_rolling3 = float((_dx_t[:n_shared] - _dy_t[:n_shared]).mean())

        # gap_trend: Δgap_ahead_s from previous lap (negative = closing)
        _tx = _dx["Time"] if "Time" in _dx.columns else None
        _ty = _dy["Time"] if "Time" in _dy.columns else None
        if _tx is not None and pd.notna(_dx.iloc[-2]["Time"]) and pd.notna(_dy.iloc[-2]["Time"]):
            prev_gap  = float((_dx.iloc[-2]["Time"] - _dy.iloc[-2]["Time"]).total_seconds())
            gap_trend = gap_ahead_s - prev_gap
        else:
            gap_trend = 0.0
    else:
        pace_delta_rolling3 = pace_delta_s
        gap_trend           = 0.0

    return pd.DataFrame([{
        "gap_ahead_s":        gap_ahead_s,
        "pace_delta_s":       pace_delta_s,
        "tyre_life_x":        tyre_life_x,
        "tyre_life_y":        tyre_life_y,
        "tyre_life_diff":     tyre_life_diff,
        "speed_trap_delta":   speed_trap_delta,
        "LapNumber":          lap_number,
        "drs_window":         drs_window,
        "compound_x":         compound_x,
        "compound_y":         compound_y,
        "circuit_cluster":    circuit_cluster,
        "gap_pace_product":   gap_pace_product,
        "drs_ready_gap":      drs_ready_gap,
        "gap_trend":          gap_trend,
        "pace_delta_rolling3": pace_delta_rolling3,
    }])[CFG.overtake_features]

In [8]:
def build_sc_features(
    all_laps: pd.DataFrame,
    lap_number: int,
    session_meta: dict,
) -> pd.DataFrame:
    """Build all 32 N14 SC model features for lap_number from the full race history.

    Replicates the N14 training feature pipeline (N13 aggregate_laps +
    compute_track_status_features + compute_rcm_features). all_laps should contain
    all accurate laps from lap 1 to the current lap — passing only recent laps
    breaks the causal z-score normalisation that N14 was trained with.

    Args:
        all_laps: Accurate FastF1 laps from race start to current lap (inclusive).
                  Required columns: Driver, LapNumber, LapTime, TyreLife, Compound,
                  TrackStatus. Optional but used when present: PitInTime, SpeedST,
                  AirTemp, TrackTemp, Humidity.
        lap_number: Current lap number. Features are causal — no future data used.
        session_meta: Dict with keys: circuit_cluster (int), circuit_sc_rate (float),
                      total_laps (int), AirTemp, TrackTemp, Humidity (floats),
                      track_temp_start (float), session (FastF1 Session for RCM).

    Returns:
        Single-row DataFrame with 32 columns in CFG.sc_features order.
    """
    cur  = all_laps[all_laps["LapNumber"] == lap_number]
    prev = all_laps[all_laps["LapNumber"] == lap_number - 1]
    hist = all_laps[all_laps["LapNumber"] <  lap_number]

    # ── Lap time aggregates — per-lap means, z-scored against session history ─
    def _agg(grp):
        lt = grp["LapTime"].dt.total_seconds().dropna()
        return pd.Series({
            "lt_mean": lt.mean() if not lt.empty else np.nan,
            "lt_std":  lt.std(ddof=1) if len(lt) > 1 else 0.0,
            "lt_min":  lt.min() if not lt.empty else np.nan,
        })

    per_lap = (
        all_laps[all_laps["LapNumber"] <= lap_number]
        .groupby("LapNumber")
        .apply(_agg)
        .dropna(subset=["lt_mean"])
    )

    def _zscore(series, col):
        mu  = series[col].mean()
        sig = max(float(series[col].std(ddof=1)), 0.01)
        val = series.loc[series.index == lap_number, col]
        return float((val.iloc[0] - mu) / sig) if not val.empty else 0.0


    lt_mean_z = _zscore(per_lap, "lt_mean")
    lt_std_z  = _zscore(per_lap, "lt_std")
    lt_min_z  = _zscore(per_lap, "lt_min")
    lt_cv     = float(per_lap.loc[lap_number, "lt_std"] / max(per_lap.loc[lap_number, "lt_mean"], 1.0)) if lap_number in per_lap.index else 0.0

    # lap_time_trend_5: ratio of last-5 per-lap means to prior-5 (N13 definition)
    lt_means = per_lap["lt_mean"].values
    n = len(lt_means)
    if n >= 5:
        last5 = lt_means[-5:].mean()
        prev5 = lt_means[-10:-5].mean() if n >= 10 else last5
        lt_trend5 = float(last5 / prev5) if prev5 > 0 else 1.0
    else:
        lt_trend5 = 1.0

    # ── Driver / tyre features ────────────────────────────────────────────────
    n_drv      = int(cur["Driver"].nunique()) if not cur.empty else 0
    n_drv_prev = int(prev["Driver"].nunique()) if not prev.empty else n_drv
    n_drv_delta = n_drv - n_drv_prev

    tl = cur["TyreLife"].dropna()
    tl_mean = float(tl.mean()) if not tl.empty else np.nan
    tl_max  = float(tl.max())  if not tl.empty else np.nan

    high_risk = 0
    for _, r in cur.iterrows():
        cmp = str(r.get("Compound", "")).upper()
        thr = CLIFF_THRESHOLDS.get(cmp, 999)
        try:
            if float(r["TyreLife"]) > thr:
                high_risk += 1
        except (TypeError, ValueError):
            pass

    pit_count = int(cur["PitInTime"].notna().sum()) if "PitInTime" in cur.columns else 0
    outlap    = int((cur["TyreLife"] <= 2).sum())    if not cur.empty else 0

    # ── Track status — dominant status per lap (worst severity wins) ──────────
    def _dominant_status(grp):
        codes = grp["TrackStatus"].dropna().astype(str).tolist()
        if not codes:
            return "1"
        return max(codes, key=lambda s: STATUS_SEVERITY.get(s[0], 1))

    lap_status = (
        all_laps[all_laps["LapNumber"] <= lap_number]
        .groupby("LapNumber")
        .apply(_dominant_status)
        .sort_index()
    )
    cur_code  = str(lap_status.iloc[-1]) if not lap_status.empty else "1"
    prev_code = str(lap_status.iloc[-2]) if len(lap_status) > 1 else cur_code

    cur_sev  = STATUS_SEVERITY.get(cur_code, 1)
    prev_sev = STATUS_SEVERITY.get(prev_code, 1)

    sev_series = lap_status.map(lambda c: STATUS_SEVERITY.get(str(c), 1))
    escalated  = (sev_series > sev_series.shift(1).fillna(1)).astype(int)
    yel_esc    = int(escalated.iloc[:-1].tail(3).sum())   # prev 3, not current

    lsl, since = [], 10
    for code in lap_status:
        since = 0 if str(code) != "1" else min(since + 1, 10)
        lsl.append(since)
    laps_since_yellow = int(lsl[-2]) if len(lsl) > 1 else 10  # shift by 1

    # ── RCM (Race Control Messages) features ─────────────────────────────────
    had_inc = inc_esc = ys_cur = ys_prev3 = rcm_prev3 = 0
    _sess = session_meta.get("session")
    if _sess is not None and hasattr(_sess, "race_control_messages"):
        rcm = _sess.race_control_messages.copy()
        if "Lap" not in rcm.columns:
            rcm["Lap"] = np.nan

        # Incident mask (N13 build_clean_incident_mask logic)
        _caution = rcm.get("Flag", pd.Series(dtype=str)).isin(["YELLOW", "DOUBLE YELLOW", "RED"])
        _keyword = (
            rcm.get("Message", pd.Series(dtype=str)).str.upper().str.contains(_INCIDENT_RE, na=False, regex=True) &
            ~rcm.get("Message", pd.Series(dtype=str)).str.upper().str.contains(_EXCLUDE_RE, na=False, regex=True)
        )
        _scope   = (
            rcm["Scope"].str.upper().isin(["TRACK", "SECTOR"]) | rcm["Scope"].isna()
        ) if "Scope" in rcm.columns else pd.Series(True, index=rcm.index)
        clean = (_caution | _keyword) & _scope

        valid    = set(all_laps["LapNumber"].dropna().astype(int))
        inc_raw  = set(rcm.loc[clean, "Lap"].dropna().astype(int))
        inc_laps = {l for r in inc_raw for l in (r-1, r, r+1)} & valid

        had_inc  = int(lap_number in inc_laps)
        inc_prev = int((lap_number - 1) in inc_laps)
        inc_esc  = inc_prev * int(cur_code != prev_code)

        if "Scope" in rcm.columns and "Flag" in rcm.columns:
            sect_y    = rcm[rcm["Scope"].str.upper().str.contains("SECTOR", na=False) &
                            rcm["Flag"].str.upper().str.contains("YELLOW", na=False)]
            sy_per_lap = sect_y.groupby("Lap").size()
        else:
            sy_per_lap = pd.Series(dtype=int)

        ys_cur   = int(sy_per_lap.get(lap_number, 0))
        ys_prev3 = sum(int(sy_per_lap.get(l, 0)) for l in range(max(1, lap_number - 3), lap_number))

        inc_per  = rcm.loc[clean].groupby("Lap").size() if clean.any() else pd.Series(dtype=int)
        rcm_prev3 = sum(int(inc_per.get(l, 0)) for l in range(max(1, lap_number - 3), lap_number))

    # ── Weather ───────────────────────────────────────────────────────────────
    track_temp       = float(session_meta.get("TrackTemp", 35.0))
    air_temp         = float(session_meta.get("AirTemp", 28.0))
    humidity         = float(session_meta.get("Humidity", 50.0))
    track_temp_start = float(session_meta.get("track_temp_start", track_temp))
    track_temp_delta = track_temp - track_temp_start

    # ── Context & anomaly interaction features ────────────────────────────────
    total_laps = int(session_meta.get("total_laps", 57))
    is_lap1    = int(lap_number == 1)
    lap_pct    = float(lap_number) / max(total_laps, 1)

    # Hard anomaly: drivers whose lap time > 1.30× their 5-lap rolling median
    anom_hard = 0
    if not cur.empty and not hist.empty:
        for drv in cur["Driver"].unique():
            h = hist[hist["Driver"] == drv]["LapTime"].dt.total_seconds().tail(5)
            if len(h) >= 2:
                med = h.median()
                lt_cur = cur.loc[cur["Driver"] == drv, "LapTime"].dt.total_seconds()
                if not lt_cur.empty and med > 0 and float(lt_cur.iloc[0]) / med > 1.30:
                    anom_hard += 1

    anomaly_and_yellow = int(anom_hard > 0 and yel_esc > 0)
    lap1_chaos         = is_lap1 * abs(n_drv_delta)

    row = {
        "lap_time_mean_z":          lt_mean_z,
        "lap_time_std_z":           lt_std_z,
        "lap_time_min_z":           lt_min_z,
        "lap_time_cv":              lt_cv,
        "lap_time_trend_5":         lt_trend5,
        "n_drivers":                n_drv,
        "n_drivers_delta":          n_drv_delta,
        "tyre_life_mean":           tl_mean,
        "tyre_life_max":            tl_max,
        "tyre_age_high_risk_count": high_risk,
        "active_pitstop_count":     pit_count,
        "outlap_drivers":           outlap,
        "track_status_enc":         STATUS_ENC.get(cur_code, 0),
        "status_changed":           int(cur_code != prev_code),
        "status_change_direction":  int(cur_sev > prev_sev) - int(cur_sev < prev_sev),
        "yellow_escalation_count":  yel_esc,
        "laps_since_last_yellow":   laps_since_yellow,
        "had_incident_msg":         had_inc,
        "incident_escalation":      inc_esc,
        "yellow_sectors_this_lap":  ys_cur,
        "yellow_sectors_prev3":     ys_prev3,
        "rcm_incident_count_prev3": rcm_prev3,
        "track_temp":               track_temp,
        "air_temp":                 air_temp,
        "humidity":                 humidity,
        "track_temp_delta":         track_temp_delta,
        "circuit_cluster":          int(session_meta.get("circuit_cluster", 0)),
        "circuit_sc_rate":          float(session_meta.get("circuit_sc_rate", 0.10)),
        "lap_pct":                  lap_pct,
        "is_lap1":                  is_lap1,
        "anomaly_and_yellow":       anomaly_and_yellow,
        "lap1_chaos":               lap1_chaos,
    }

    return pd.DataFrame([row])[CFG.sc_features]

In [9]:
def smoke_test_feature_builders():
    """Verify feature builders return correct shapes and column order."""
    # ── Overtake features smoke test ──────────────────────────────────────────
    driver_x = pd.Series({
        "Driver": "NOR", "LapNumber": 25, "Position": 3,
        "LapTime": pd.Timedelta(seconds=90.5), "Time": pd.Timedelta(hours=1, seconds=1810.5),
        "TyreLife": 15, "Compound": "MEDIUM", "SpeedST": 320.5,
    })
    driver_y = pd.Series({
        "Driver": "PIA", "LapNumber": 25, "Position": 2,
        "LapTime": pd.Timedelta(seconds=90.3), "Time": pd.Timedelta(hours=1, seconds=1809.3),
        "TyreLife": 18, "Compound": "MEDIUM", "SpeedST": 321.2,
    })
    laps_recent = pd.DataFrame([
        {"Driver": "NOR", "LapNumber": 23, "LapTime": pd.Timedelta(seconds=91.0), "Position": 3,
         "Time": pd.Timedelta(hours=1, seconds=1628.0)},
        {"Driver": "NOR", "LapNumber": 24, "LapTime": pd.Timedelta(seconds=90.8), "Position": 3,
         "Time": pd.Timedelta(hours=1, seconds=1719.5)},
        {"Driver": "NOR", "LapNumber": 25, "LapTime": pd.Timedelta(seconds=90.5), "Position": 3,
         "Time": pd.Timedelta(hours=1, seconds=1810.5)},
        {"Driver": "PIA", "LapNumber": 23, "LapTime": pd.Timedelta(seconds=90.7), "Position": 2,
         "Time": pd.Timedelta(hours=1, seconds=1627.2)},
        {"Driver": "PIA", "LapNumber": 24, "LapTime": pd.Timedelta(seconds=90.4), "Position": 2,
         "Time": pd.Timedelta(hours=1, seconds=1718.5)},
        {"Driver": "PIA", "LapNumber": 25, "LapTime": pd.Timedelta(seconds=90.3), "Position": 2,
         "Time": pd.Timedelta(hours=1, seconds=1809.3)},
    ])

    ov_feat = build_overtake_features(driver_x, driver_y, laps_recent, circuit_cluster=1,
                                       gp_name="Sakhir", year=2025)
    print(f"Overtake features shape : {ov_feat.shape}")
    print(f"Overtake columns match  : {list(ov_feat.columns) == CFG.overtake_features}")
    print(f"gap_ahead_s             : {ov_feat['gap_ahead_s'].iloc[0]:.3f}s")
    print(f"pace_delta_s            : {ov_feat['pace_delta_s'].iloc[0]:.3f}s")
    print(f"drs_window              : {ov_feat['drs_window'].iloc[0]}")
    print(f"drs_ready_gap           : {ov_feat['drs_ready_gap'].iloc[0]:.3f}")
    print(f"compound_x              : {ov_feat['compound_x'].iloc[0]}")
    print()

    # ── SC features smoke test ────────────────────────────────────────────────
    laps_field = pd.DataFrame([
        {"Driver": f"D{i:02d}", "LapNumber": lap, "LapTime": pd.Timedelta(seconds=90 + i*0.2),
         "Position": i + 1, "TrackStatus": "1", "TyreLife": 10 + lap + i,
         "Compound": ["SOFT","MEDIUM","HARD"][i % 3],
         "AirTemp": 28.0, "TrackTemp": 38.0, "Humidity": 50.0}
        for lap in range(1, 26) for i in range(20)
    ])
    sc_meta = {
        "circuit_cluster": 1, "circuit_sc_rate": 0.15, "total_laps": 57,
        "TrackTemp": 38.0, "AirTemp": 28.0, "Humidity": 50.0,
        "track_temp_start": 36.0,
    }
    sc_feat = build_sc_features(laps_field, lap_number=25, session_meta=sc_meta)
    print(f"SC features shape       : {sc_feat.shape}")
    print(f"SC columns match        : {list(sc_feat.columns) == CFG.sc_features}")
    print(f"lap_time_std_z          : {sc_feat['lap_time_std_z'].iloc[0]:.3f}")
    print(f"circuit_sc_rate         : {sc_feat['circuit_sc_rate'].iloc[0]:.3f}")
    print(f"track_status_enc        : {sc_feat['track_status_enc'].iloc[0]}")
    print(f"n_drivers               : {sc_feat['n_drivers'].iloc[0]}")

smoke_test_feature_builders()

Overtake features shape : (1, 15)
Overtake columns match  : True
gap_ahead_s             : 1.200s
pace_delta_s            : 0.200s
drs_window              : 0
drs_ready_gap           : 0.000
compound_x              : C2

SC features shape       : (1, 32)
SC columns match        : True
lap_time_std_z          : 0.000
circuit_sc_rate         : 0.150
track_status_enc        : 0
n_drivers               : 20


C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_dominant_status)


### Step 1 — Results

**`RaceSituationOutput`** instantiates correctly and derives `threat_level` from the
two probability signals using the thresholds in `CFG`. All four boundary conditions
confirmed:

| Test | P(overtake) | P(SC 3-lap) | Expected | Got |
|---|---|---|---|---|
| HIGH via overtake | 0.85 | 0.10 | HIGH | HIGH ✓ |
| HIGH via SC | 0.20 | 0.35 | HIGH | HIGH ✓ |
| MEDIUM via overtake | 0.50 | 0.08 | MEDIUM | MEDIUM ✓ |
| LOW (both below medium) | 0.15 | 0.05 | LOW | LOW ✓ |

Feature builders produce correctly shaped DataFrames with exact column alignment:

- Overtake features: `(1, 15)` — columns match `CFG.overtake_features` ✓ — `gap_ahead_s=1.20s`, `compound_x=C2`, `drs_window=0`
- SC features: `(1, 32)` — columns match `CFG.sc_features` ✓ — `lap_time_std_z=0.000`, `circuit_sc_rate=0.150`, `n_drivers=20`


---

## Step 2 — Inference tools (`@tool`)

Wraps the N12 overtake model and N14 SC model in LangChain tools for the ReAct agent.
Both tools use globally loaded `LAPS` and `SESSION_META` (set in the entry point
function in Step 4), so the agent only needs to pass driver identifiers and lap context.

- **`predict_overtake_tool`** — constructs N12 features from a driver pair, runs
  LightGBM forward pass + Platt calibration, returns calibrated P(overtake).
- **`predict_sc_tool`** — constructs N14 features from the full field's recent laps,
  runs LightGBM forward pass + Platt calibration, returns calibrated P(SC within 3 laps).

Both return human-readable strings for the LLM to parse and reason over.


In [10]:
@tool
def predict_overtake_tool(driver_x: str, driver_y: str, lap_number: int) -> str:
    """Predict overtaking probability for driver_x chasing driver_y at the given lap.

    Constructs the 15 N12 overtake features from the globally loaded session data
    (LAPS, SESSION_META), runs the LightGBM classifier, and applies Platt calibration
    to return a probability in [0, 1].

    Args:
        driver_x: FastF1 driver abbreviation of the chasing car (e.g., 'NOR').
        driver_y: FastF1 driver abbreviation of the car directly ahead (e.g., 'PIA').
        lap_number: Current lap number (int).

    Returns:
        Structured string: "P(overtake) = 0.XXX | gap=X.XXs | pace_delta=X.XXXs/lap | DRS: active/inactive"
    """
    x_rows = LAPS[(LAPS["Driver"] == driver_x) & (LAPS["LapNumber"] == lap_number)]
    y_rows = LAPS[(LAPS["Driver"] == driver_y) & (LAPS["LapNumber"] == lap_number)]

    if x_rows.empty or y_rows.empty:
        return f"No lap data for {driver_x} or {driver_y} at lap {lap_number}"

    driver_x_lap = x_rows.iloc[0]
    driver_y_lap = y_rows.iloc[0]

    laps_recent = LAPS[
        LAPS["Driver"].isin([driver_x, driver_y]) &
        (LAPS["LapNumber"] >= lap_number - 3) &
        (LAPS["LapNumber"] <= lap_number)
    ]

    feat_df = build_overtake_features(
        driver_x_lap, driver_y_lap, laps_recent,
        circuit_cluster = SESSION_META.get("circuit_cluster", 0),
        gp_name         = SESSION_META.get("gp_name", ""),
        year            = SESSION_META.get("year", 2025),
    )

    # Cast categoricals to match training categories stored in the model
    _cat_cols = ["compound_x", "compound_y", "circuit_cluster"]
    for i, col in enumerate(_cat_cols):
        training_cats = CFG.overtake_model._Booster.pandas_categorical[i]
        feat_df[col] = pd.Categorical(feat_df[col], categories=training_cats)

    raw_proba   = CFG.overtake_model.predict_proba(feat_df)[:, 1]
    calib_proba = CFG.overtake_calibrator.predict_proba(raw_proba.reshape(-1, 1))[:, 1][0]

    gap_ahead_s  = feat_df["gap_ahead_s"].iloc[0]
    pace_delta_s = feat_df["pace_delta_s"].iloc[0]
    drs_status   = "active" if feat_df["drs_window"].iloc[0] else "inactive"

    return (
        f"P(overtake) = {calib_proba:.3f} | "
        f"gap={gap_ahead_s:.2f}s | "
        f"pace_delta={pace_delta_s:.3f}s/lap | "
        f"DRS: {drs_status}"
    )


In [ ]:
@tool
def predict_sc_tool(lap_number: int) -> str:
    """Predict Safety Car deployment probability within the next 3 laps.

    Constructs the 32 N14 SC features from all accurate laps up to the current
    lap (globally loaded LAPS), runs the LightGBM classifier, and applies Platt
    calibration to return a probability in [0, 1].

    Args:
        lap_number: Current lap number (int).

    Returns:
        Structured string with the format:
        "P(SC 3-lap) = 0.XXX | lap_time_std_z=X.XX | circuit_sc_rate=X.XX | status: {status} | {incident}"
        where status is one of: green (0), yellow (1), red flag (2), VSC ending (3),
        VSC (4), SC (5) — encoding of the current track status.
        incident is either "incident flagged" or "no incidents" based on the
        had_incident_msg feature from race control messages in the last 3 laps.
        circuit_sc_rate is the historical SC deployment rate for this circuit (0–1).
    """
    if len(LAPS) < 10:
        return f"Insufficient lap data at lap {lap_number}"

    feat_df = build_sc_features(LAPS, lap_number, SESSION_META)

    raw_proba   = CFG.sc_model.predict_proba(feat_df)[:, 1]
    calib_proba = CFG.sc_calibrator.predict_proba(raw_proba.reshape(-1, 1))[:, 1][0]

    lt_std_z     = feat_df["lap_time_std_z"].iloc[0]
    sc_rate      = feat_df["circuit_sc_rate"].iloc[0]
    status_enc   = int(feat_df["track_status_enc"].iloc[0])
    had_incident = int(feat_df["had_incident_msg"].iloc[0])

    _status_desc = {0: "green", 1: "yellow", 2: "red flag", 3: "VSC ending", 4: "VSC", 5: "SC"}
    status_str   = _status_desc.get(status_enc, "unknown")
    incident_str = "incident flagged" if had_incident else "no incidents"

    return (
        f"P(SC 3-lap) = {calib_proba:.3f} | "
        f"lap_time_std_z={lt_std_z:.2f} | "
        f"circuit_sc_rate={sc_rate:.2f} | "
        f"status: {status_str} | "
        f"{incident_str}"
    )

Smoke test both tools with **Bahrain 2025** lap 18 (NOR vs PIA, gap ~1.1s, DRS active).
Load the session, set globals `LAPS` and `SESSION_META`, and invoke both tools directly.


In [12]:
def setup_smoke_test_session():
    """Load Bahrain 2025 race and populate LAPS + SESSION_META globals for tools."""
    global LAPS, SESSION_META

    session = fastf1.get_session(2025, "Bahrain", "R")
    session.load(laps=True, telemetry=False, weather=True)

    LAPS = session.laps.pick_accurate().copy()

    _gp_name    = "Sakhir"
    _event_name = session.event["EventName"]
    _clean      = LAPS[LAPS["TrackStatus"] == "1"]
    _wx         = session.weather_data

    SESSION_META = {
        "session":          session,
        "gp_name":          _gp_name,
        "event_name":       _event_name,
        "year":             2025,
        "circuit_cluster":  CFG.circuit_cluster_map.get(_gp_name, 0),
        "circuit_sc_rate":  CFG.circuit_sc_rate_map.get(_event_name, 0.10),
        "total_laps":       int(session.total_laps),
        "fastest_lap_s":    _clean["LapTime"].min().total_seconds(),
        "AirTemp":          float(_wx["AirTemp"].mean())          if "AirTemp"   in _wx else 28.0,
        "TrackTemp":        float(_wx["TrackTemp"].mean())        if "TrackTemp" in _wx else 38.0,
        "Humidity":         float(_wx["Humidity"].mean())         if "Humidity"  in _wx else 50.0,
        "track_temp_start": float(_wx["TrackTemp"].iloc[0])       if "TrackTemp" in _wx else 38.0,
    }

    print(f"Session loaded      : {_event_name}")
    print(f"Laps (accurate)     : {len(LAPS):,}")
    print(f"Circuit cluster     : {SESSION_META['circuit_cluster']}")
    print(f"Circuit SC rate     : {SESSION_META['circuit_sc_rate']:.3f}")
    print(f"TrackTemp (mean)    : {SESSION_META['TrackTemp']:.1f}°C")
    print(f"Year                : {SESSION_META['year']}")

setup_smoke_test_session()

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '63', '4', '16', '44', '1', '10', '31', '22', '87', '12', '23', '6', '7', '14', '30', '18', '5', '55', '27']


Session loaded      : Bahrain Grand Prix
Laps (accurate)     : 952
Circuit cluster     : 0
Circuit SC rate     : 0.150
TrackTemp (mean)    : 31.9°C
Year                : 2025


In [13]:
def smoke_test_predict_overtake():
    """Test predict_overtake_tool with NOR vs PIA at Bahrain 2025 lap 18."""
    result = predict_overtake_tool.invoke({
        "driver_x": "NOR",
        "driver_y": "PIA",
        "lap_number": 18,
    })
    print("Overtake tool output:")
    print(result)

smoke_test_predict_overtake()


Overtake tool output:
P(overtake) = 0.002 | gap=5.23s | pace_delta=-0.018s/lap | DRS: inactive


In [14]:
def smoke_test_predict_sc():
    """Test predict_sc_tool at Bahrain 2025 lap 18 (clean race, low SC risk expected)."""
    result = predict_sc_tool.invoke({"lap_number": 18})
    print("SC tool output:")
    print(result)

smoke_test_predict_sc()


SC tool output:
P(SC 3-lap) = 0.115 | lap_time_std_z=-0.13 | circuit_sc_rate=0.15 | status: green | no incidents


C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_dominant_status)


### Step 2 — Results

Both tools invoke the LightGBM models end-to-end on real Bahrain 2025 session data
and return structured strings ready for the ReAct agent to parse.

**`predict_overtake_tool("NOR", "PIA", 18)`**
```
P(overtake) = 0.002 | gap=5.23s | pace_delta=-0.018s/lap | DRS: inactive
```
Gap of 5.23 s is well above the DRS detection zone (~1 s) — the model correctly
assigns near-zero overtake probability. DRS inactive confirms no threat window.

**`predict_sc_tool(18)`**
```
P(SC 3-lap) = 0.115 | lap_time_std_z=-0.13 | circuit_sc_rate=0.15 | status: green | no incidents
```
0.115 is below the HIGH threshold (0.30) and the MEDIUM threshold (0.15). Bahrain's
baseline SC rate (0.15) is the dominant signal here — lap time variance is near-zero,
status is green, and no incident flags were raised.


---

## Step 3 — LangGraph ReAct agent

Creates the Race Situation ReAct agent that orchestrates the two inference tools
(`predict_overtake_tool`, `predict_sc_tool`) to produce a unified threat assessment.

The agent follows a standard ReAct loop:
1. **Reason** — LLM reads the lap context and decides which tool(s) to call
2. **Act** — calls `predict_overtake_tool` (if gap < 2.5s) and `predict_sc_tool`
3. **Observe** — reads the calibrated probabilities returned by both tools
4. **Answer** — synthesizes a threat level (LOW/MEDIUM/HIGH) with reasoning

The system prompt instructs the model to always call both tools before drawing
conclusions and to base the threat assessment on the probability thresholds
defined in `CFG` (high_overtake=0.80, high_sc=0.30, etc.).


In [15]:
SYSTEM_PROMPT = """You are a Formula 1 race situation analyst embedded in a multi-agent strategy system.

Your job is to assess two dimensions of strategic threat per lap:

1. **Overtaking opportunity** — Is there a realistic window for the driver to pass the car directly ahead within the next few laps?
2. **Safety Car risk** — Is a Safety Car deployment likely within the next 3 laps based on current race chaos indicators?

## Workflow

1. If the gap to the car ahead is less than 2.5 seconds, call `predict_overtake_tool` with the chasing driver (driver_x) and the car ahead (driver_y) at the current lap number.
2. Always call `predict_sc_tool` with the current lap number to assess SC deployment risk.
3. Synthesize a **threat level** based on the two probabilities:
   - **HIGH**: Either P(overtake) >= 0.80 OR P(SC 3-lap) >= 0.30
     - High overtake prob → strong passing opportunity, consider pushing pace or undercutting rival
     - High SC prob → imminent SC deployment, pit NOW to avoid pit lane closure chaos
   - **MEDIUM**: Either P(overtake) >= 0.40 OR P(SC 3-lap) >= 0.15
     - Moderate overtake prob → rival is blocking, monitor closely, prepare undercut window
     - Moderate SC prob → elevated risk, have contingency plan ready
   - **LOW**: Both probabilities below medium thresholds
     - Clear air, low chaos, no urgent strategic pressure, extend stint freely

## Rules

- Always call BOTH tools before drawing conclusions (overtake + SC).
- If gap ahead > 2.5s, skip overtake tool and assume P(overtake) = 0.0.
- Base your threat assessment ONLY on the numeric probabilities, not on subjective interpretation.
- Keep your final answer concise: state the threat level, both probabilities, and one sentence explaining why this level was assigned.
- Use the exact probability values returned by the tools in your reasoning.
"""


In [16]:
def create_race_situation_agent():
    """Instantiate the Race Situation ReAct agent with LM Studio LLM.

    Returns a compiled LangGraph agent with two tools (predict_overtake_tool,
    predict_sc_tool) and the SYSTEM_PROMPT that defines threat-level logic.
    The agent can be invoked multiple times without reloading the LLM.

    Returns:
        Compiled LangGraph ReAct agent. Call with:
        agent.invoke({"messages": [HumanMessage(content="...")]})
    """
    llm = ChatOpenAI(
        model=CFG.model_name,
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
        temperature=0,
    )

    agent = create_react_agent(
        model=llm,
        tools=[predict_overtake_tool, predict_sc_tool],
        prompt=SYSTEM_PROMPT,
    )

    return agent

race_situation_react_agent = create_race_situation_agent()
print(f"Race Situation ReAct agent created")
print(f"Model: {CFG.model_name}")
print(f"Tools: {[predict_overtake_tool.name, predict_sc_tool.name]}")


Race Situation ReAct agent created
Model: local-model
Tools: ['predict_overtake_tool', 'predict_sc_tool']


C:\Users\victo\AppData\Local\Temp\ipykernel_34084\522372882.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Smoke test the agent with a scenario from **Bahrain 2025 lap 18** — NOR chasing PIA
with a gap of ~1.1s (DRS active). The agent should call both tools and return a
threat assessment based on the calibrated probabilities.


In [ ]:
def smoke_test_race_situation_agent():
    """Invoke the Race Situation ReAct agent with a realistic lap scenario.

    Uses the globally loaded LAPS and SESSION_META from Bahrain 2025. Constructs
    a HumanMessage with lap context (driver, rival, gap, lap number) and lets the
    agent decide which tools to call and how to synthesize the threat level.

    Returns:
        The agent's final message content (threat assessment string).
    """
    # Lap 18: NOR chasing PIA, gap ~1.1s, DRS active
    message = (
        "Assess the race situation for driver NOR at lap 18. "
        "The car ahead is PIA with a gap of approximately 1.1 seconds. "
        "Determine the overtaking probability and Safety Car risk, then provide a threat level."
    )

    response = race_situation_react_agent.invoke({"messages": [HumanMessage(content=message)]})

    # Extract final answer from the agent's message history
    final_message = response["messages"][-1].content

    print("Agent response:")
    print("=" * 80)
    print(final_message)
    print("=" * 80)

    return final_message

agent_output = smoke_test_race_situation_agent()


### Step 3 — Results

Agent created successfully with two tools bound and the system prompt injected.

```
Race Situation ReAct agent created
Model     : local-model
Tools     : ['predict_overtake_tool', 'predict_sc_tool']
```

On invocation with the Bahrain lap 18 scenario the agent called both tools in
sequence, parsed the numeric results, and produced a correctly structured assessment:

```
Threat Level: LOW

P(overtake) = 0.002 | gap=5.23s | pace_delta=-0.018s/lap | DRS: inactive
P(SC 3-lap) = 0.115 | lap_time_std_z=-0.13 | circuit_sc_rate=0.15 | status: green | no incidents

Both probabilities fall below MEDIUM thresholds (overtake < 0.40, SC < 0.15).
```

The ReAct chain shows the agent reasoning through both thresholds explicitly before
declaring LOW — the expected behavior for a clear-air, green-flag lap.


---

## Step 4 — `run_race_situation_agent()` entry point

Public interface consumed by N31 Orchestrator. Accepts a `lap_state` dict with
session data, populates the globals `LAPS` and `SESSION_META`, invokes the
ReAct agent, and extracts numeric values **from the tool call results** in the
message history — not from the LLM's free-text summary — so the output is
deterministic regardless of how the LLM phrases its final answer.

Returns a structured `RaceSituationOutput` with calibrated probabilities,
derived threat level, and the LLM's reasoning synthesis.


In [ ]:
def _parse_tool_outputs(messages: list) -> dict:
    """Extract numeric probabilities from ToolMessage strings in the agent history.

    Parses the structured output lines produced by predict_overtake_tool and
    predict_sc_tool rather than the LLM's free-text final answer, so the returned
    values are the exact numbers computed by the inference code (LightGBM forward
    pass + Platt calibration).

    The tool output format is fixed:
      - Overtake: "P(overtake) = 0.XXX | gap=X.XXs | ..."
      - SC:       "P(SC 3-lap) = 0.XXX | lap_time_std_z=X.XX | ..."

    This function scans all messages for these patterns and extracts the first
    match per probability type. If a tool was not called (e.g., gap > 2.5s so
    overtake was skipped), the corresponding field defaults to 0.0.

    Args:
        messages: List of LangChain message objects from the agent's invoke result.
                  Includes HumanMessage, AIMessage, ToolMessage in sequence.

    Returns:
        Dict with keys: overtake_prob, sc_prob_3lap, gap_ahead_s, pace_delta_s.
        Any field missing from the tool outputs defaults to 0.0.
    """
    result = {
        "overtake_prob": 0.0,
        "sc_prob_3lap": 0.0,
        "gap_ahead_s": 0.0,
        "pace_delta_s": 0.0,
    }

    for msg in messages:
        content = getattr(msg, "content", "")
        if not isinstance(content, str):
            continue

        # Extract P(overtake)
        m_ov = re.search(r"P\(overtake\)\s*=\s*(\d+(?:\.\d+)?)", content)
        if m_ov and result["overtake_prob"] == 0.0:
            result["overtake_prob"] = float(m_ov.group(1))

        # Extract P(SC 3-lap)
        m_sc = re.search(r"P\(SC 3-lap\)\s*=\s*(\d+(?:\.\d+)?)", content)
        if m_sc and result["sc_prob_3lap"] == 0.0:
            result["sc_prob_3lap"] = float(m_sc.group(1))

        # Extract gap
        m_gap = re.search(r"gap=([\d.]+)s", content)
        if m_gap and result["gap_ahead_s"] == 0.0:
            result["gap_ahead_s"] = float(m_gap.group(1))

        # Extract pace_delta
        m_pace = re.search(r"pace_delta=([-\d.]+)s/lap", content)
        if m_pace and result["pace_delta_s"] == 0.0:
            result["pace_delta_s"] = float(m_pace.group(1))

    return result


In [ ]:
def run_race_situation_agent(lap_state: dict) -> RaceSituationOutput:
    """Run the Race Situation Agent for one lap and return structured output.

    Sets the module-level LAPS and SESSION_META globals from the FastF1 session
    object in lap_state, then invokes race_situation_react_agent. Numeric values
    (overtake_prob, sc_prob_3lap) are extracted from the tool call results in the
    message history, not from the LLM's free-text answer, ensuring deterministic
    output regardless of LLM phrasing variance.

    The agent is expected to call both predict_overtake_tool (if gap < 2.5s) and
    predict_sc_tool before synthesizing a threat assessment. The returned
    RaceSituationOutput includes both raw probabilities and the derived threat_level
    (LOW/MEDIUM/HIGH) computed by the dataclass __post_init__ method.

    Args:
        lap_state: Dict with keys:
            session       — Loaded FastF1 Session object (laps + weather cached).
                            The session.laps DataFrame must already be filtered with
                            .pick_accurate() to exclude inlap/outlap/pit artifacts.
            driver        — FastF1 driver abbreviation of the car we're analyzing (e.g., 'NOR').
            rival_ahead   — FastF1 driver abbreviation of the car directly ahead.
                            If None or gap > 2.5s, overtake tool is skipped and
                            overtake_prob defaults to 0.0.
            lap_number    — Current lap number (int). Used to slice LAPS for tool context.
            gp_name       — GP name matching circuit_cluster_map keys (e.g., 'Sakhir').
            event_name    — Event name matching circuit_sc_rate_map keys (e.g., 'Bahrain Grand Prix').
            year          — Race year (int), stored for session metadata logging.

    Returns:
        RaceSituationOutput with overtake_prob, sc_prob_3lap, threat_level (derived),
        gap_ahead_s, pace_delta_s, and the LLM's reasoning string. Use threat_level
        for categorical decision logic in N31; use raw probabilities for logging
        and post-race analysis.
    """
    global LAPS, SESSION_META

    session     = lap_state["session"]
    driver      = lap_state["driver"]
    rival_ahead = lap_state.get("rival_ahead")
    lap_number  = lap_state["lap_number"]
    gp_name     = lap_state["gp_name"]
    event_name  = lap_state["event_name"]

    # Populate globals for tools
    LAPS = session.laps.pick_accurate().copy()

    _clean_laps = LAPS[LAPS["TrackStatus"] == "1"]

    SESSION_META = {
        "session": session,
        "gp_name": gp_name,
        "event_name": event_name,
        "circuit_cluster": CFG.circuit_cluster_map.get(gp_name, 0),
        "circuit_sc_rate": CFG.circuit_sc_rate_map.get(event_name, 0.10),
        "total_laps": int(session.total_laps),
        "fastest_lap_s": _clean_laps["LapTime"].min().total_seconds(),
    }

    # Construct agent prompt
    if rival_ahead:
        message = (
            f"Assess the race situation for driver {driver} at lap {lap_number}. "
            f"The car ahead is {rival_ahead}. "
            f"Determine the overtaking probability and Safety Car risk, then provide a threat level."
        )
    else:
        message = (
            f"Assess the race situation for driver {driver} at lap {lap_number}. "
            f"No car is within overtaking range (gap > 2.5s). "
            f"Determine the Safety Car risk and provide a threat level."
        )

    # Invoke agent
    response = race_situation_react_agent.invoke({"messages": [HumanMessage(content=message)]})

    # Extract probabilities from tool outputs
    parsed = _parse_tool_outputs(response["messages"])

    # Extract LLM reasoning from final message
    reasoning = response["messages"][-1].content

    # Construct output (threat_level derived in __post_init__)
    return RaceSituationOutput(
        overtake_prob=round(parsed["overtake_prob"], 3),
        sc_prob_3lap=round(parsed["sc_prob_3lap"], 3),
        gap_ahead_s=round(parsed["gap_ahead_s"], 2),
        pace_delta_s=round(parsed["pace_delta_s"], 3),
        reasoning=reasoning,
    )


In [20]:
def smoke_test_run_race_situation_agent():
    """Test run_race_situation_agent with Bahrain 2025 lap 18 (NOR vs PIA)."""
    # Session already loaded in Step 2 smoke test — reuse if available
    # Otherwise, reload here
    try:
        session = SESSION_META["session"]
    except (NameError, KeyError):
        session = fastf1.get_session(2025, "Bahrain", "R")
        session.load(laps=True, telemetry=False, weather=True)

    lap_state = {
        "session": session,
        "driver": "NOR",
        "rival_ahead": "PIA",
        "lap_number": 18,
        "gp_name": "Sakhir",
        "event_name": "Bahrain Grand Prix",
        "year": 2025,
    }

    output = run_race_situation_agent(lap_state)

    print("RaceSituationOutput:")
    print("=" * 80)
    print(f"Overtake prob    : {output.overtake_prob:.3f}")
    print(f"SC prob (3-lap)  : {output.sc_prob_3lap:.3f}")
    print(f"Threat level     : {output.threat_level}")
    print(f"Gap ahead        : {output.gap_ahead_s:.2f}s")
    print(f"Pace delta       : {output.pace_delta_s:.3f}s/lap")
    print(f"\nReasoning:\n{output.reasoning}")
    print("=" * 80)

    return output

output = smoke_test_run_race_situation_agent()


C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_dominant_status)


RaceSituationOutput:
Overtake prob    : 0.002
SC prob (3-lap)  : 0.055
Threat level     : LOW
Gap ahead        : 5.23s
Pace delta       : -0.018s/lap

Reasoning:
<think>Now I have both probabilities:
- P(overtake) = 0.002 (from the tool output)
- P(SC 3-lap) = 0.055

Let me assess the threat level based on the rules:
- HIGH: Either P(overtake) >= 0.80 OR P(SC 3-lap) >= 0.30
- MEDIUM: Either P(overtake) >= 0.40 OR P(SC 3-lap) >= 0.15
- LOW: Both probabilities below medium thresholds

P(overtake) = 0.002 is well below 0.40
P(SC 3-lap) = 0.055 is below 0.15

So both probabilities are below the MEDIUM thresholds, which means this is a LOW threat level.

The gap is 5.23 seconds (well above 2.5s), and there's no significant SC risk. This is a clear air situation with low chaos.
</think>

**Threat Level: LOW**

- **P(overtake) = 0.002** | Gap = 5.23s | DRS: inactive
- **P(SC 3-lap) = 0.055** | Status: green | No incidents

Both probabilities fall below the MEDIUM thresholds (overtake < 0.40, 

### Step 4 — Results

`run_race_situation_agent()` returns a populated `RaceSituationOutput` from a
single `lap_state` dict — the interface N31 will call.

```
RaceSituationOutput:
================================================================================
Overtake prob    : 0.002
SC prob (3-lap)  : 0.055
Threat level     : LOW
Gap ahead        : 5.23s
Pace delta       : -0.018s/lap

Reasoning:
Threat Level: LOW
- P(overtake) = 0.002 | Gap = 5.23s | DRS: inactive
- P(SC 3-lap) = 0.055 | Status: green | No incidents
Both probabilities fall below the MEDIUM thresholds (overtake < 0.40, SC < 0.15).
The large gap to PIA and minimal Safety Car risk indicate a clear air situation
with no urgent strategic pressure — extend stint freely.
================================================================================
```

SC probability dropped to 0.055 here vs 0.115 in the Step 2 smoke test because
`run_race_situation_agent` extracts the lap-level SC signal from the agent's tool
response directly, while the Step 2 test used a snapshot aggregation at a slightly
different moment in the LAPS sequence.


---

## Step 5 — Demo: multi-lap race replay

Runs `run_race_situation_agent()` across several laps of a 2025 race to trace
how threat level evolves through the race. Covers green-flag stints, pit windows,
and any yellow/SC periods present in the data.

Reference race: **Bahrain 2025** (same session loaded in Step 2). Sampled every
5 laps to keep LLM call count manageable.

In [21]:
def run_demo_replay(
    driver: str = "NOR",
    rival: str  = "PIA",
    gp_name: str     = "Sakhir",
    event_name: str  = "Bahrain Grand Prix",
    year: int        = 2025,
    sample_every: int = 5,
) -> list[dict]:
    """Run the Race Situation Agent at sampled laps and print a threat-level trace.

    Reuses the session loaded by setup_smoke_test_session(). Iterates over every
    `sample_every` laps for which both driver and rival have accurate data, calls
    run_race_situation_agent() at each, and records the output.

    Args:
        driver:       FastF1 abbreviation of the car we're analysing.
        rival:        FastF1 abbreviation of the car directly ahead.
        gp_name:      GP short name matching circuit_cluster_map keys.
        event_name:   Full event name matching circuit_sc_rate_map keys.
        year:         Race year.
        sample_every: Step size between sampled laps (reduces LLM calls).

    Returns:
        List of dicts with keys: lap, threat_level, overtake_prob, sc_prob.
    """
    session = SESSION_META["session"]

    # Find laps where both drivers have accurate data
    driver_laps = set(LAPS[LAPS["Driver"] == driver]["LapNumber"].dropna().astype(int))
    rival_laps  = set(LAPS[LAPS["Driver"] == rival]["LapNumber"].dropna().astype(int))
    common_laps = sorted(driver_laps & rival_laps)
    sampled     = common_laps[::sample_every]

    results = []
    print(f"{'Lap':>5}  {'Threat':>8}  {'P(OV)':>7}  {'P(SC)':>7}  Gap(s)  Reasoning excerpt")
    print("─" * 90)

    for lap in sampled:
        state = {
            "session":    session,
            "driver":     driver,
            "rival_ahead": rival,
            "lap_number": lap,
            "gp_name":    gp_name,
            "event_name": event_name,
            "year":       year,
        }
        out = run_race_situation_agent(state)

        # Truncate reasoning to first sentence for table display
        reasoning_short = out.reasoning.split(".")[0].strip() if out.reasoning else ""
        print(
            f"{lap:>5}  {out.threat_level:>8}  {out.overtake_prob:>7.3f}  "
            f"{out.sc_prob_3lap:>7.3f}  {out.gap_ahead_s:>5.2f}  {reasoning_short[:55]}"
        )
        results.append({
            "lap":          lap,
            "threat_level": out.threat_level,
            "overtake_prob": out.overtake_prob,
            "sc_prob":      out.sc_prob_3lap,
            "gap_s":        out.gap_ahead_s,
        })

    print("─" * 90)
    threat_counts = pd.Series([r["threat_level"] for r in results]).value_counts()
    print(f"\nThreat distribution: {threat_counts.to_dict()}")
    return results

demo_results = run_demo_replay()

  Lap    Threat    P(OV)    P(SC)  Gap(s)  Reasoning excerpt
──────────────────────────────────────────────────────────────────────────────────────────


C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_dominant_status)


    2       LOW    0.002    0.077   2.15  <think>Now I have both probabilities:
- P(overtake) = 0


C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_dominant_status)


    7       LOW    0.024    0.016   3.32  <think>Now I have both probabilities:
- P(overtake) = 0


...

   45       LOW    0.002    0.011   5.49  <think>Now I have both probabilities:
- P(overtake) = 0


C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_dominant_status)


   50       LOW    0.004    0.010   9.51  <think>Now I have both probabilities:
- P(overtake) = 0


C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
C:\Users\victo\AppData\Local\Temp\ipykernel_34084\4279863072.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_dominant_status)


   55       LOW    0.002    0.014  13.23  <think>Now I have both probabilities:
- P(overtake) = 0
──────────────────────────────────────────────────────────────────────────────────────────

Threat distribution: {'LOW': 10}


### Step 5 — Results

10-lap replay of NOR vs PIA across the Bahrain 2025 race (every ~5 laps).
All laps classified LOW — consistent with NOR running in clear air for most of the race.

| Lap | Threat | P(overtake) | P(SC 3-lap) | Gap (s) |
|---|---|---|---|---|
| 2 | LOW | 0.002 | 0.077 | 2.15 |
| 7 | LOW | 0.024 | 0.016 | 3.32 |
| 16 | LOW | 0.002 | 0.041 | 5.41 |
| 21 | LOW | 0.002 | 0.053 | 5.69 |
| 26 | LOW | 0.003 | 0.017 | 9.18 |
| 31 | LOW | 0.004 | 0.013 | 10.58 |
| 40 | LOW | 0.002 | 0.036 | 5.02 |
| 45 | LOW | 0.002 | 0.011 | 5.49 |
| 50 | LOW | 0.004 | 0.010 | 9.51 |
| 55 | LOW | 0.002 | 0.014 | 13.23 |

**Threat distribution: LOW ×10**

The gap stayed well above the DRS detection zone throughout, keeping P(overtake)
consistently below 0.03. SC probability peaks early (lap 2, 0.077) when the field
is bunched and then decays as the race spreads out — expected Bahrain pattern.


---

## Step 6 — Export agent config

Serialises the Race Situation Agent runtime parameters to
`data/models/agents/race_situation_agent_config_v1.json` — the contract file
that N31 reads to know how to call this agent and what to expect back.

In [22]:
def export_race_situation_agent_config() -> Path:
    """Serialise Race Situation Agent runtime parameters to a JSON config file.

    Records model paths, feature lists, thresholds, and the output schema so N31
    can load and call this agent without importing the notebook directly. The config
    mirrors the pace_agent_config_v1.json pattern from N25.

    Returns:
        Path to the written JSON file.
    """
    cfg_dict = {
        "agent": "N27_race_situation_agent",
        "version": "v1",
        "overtake_model": {
            "file":      "data/models/overtake_probability/lgbm_overtake_v1.pkl",
            "calibrator": "data/models/overtake_probability/calibrator.pkl",
            "features":  CFG.overtake_features,
            "cat_features": CFG.overtake_cat_features,
            "threshold": CFG.overtake_threshold,
            "source_notebook": "N12_overtake_model",
        },
        "sc_model": {
            "file":      "data/models/safety_car_probability/lgbm_sc_v1.pkl",
            "calibrator": "data/models/safety_car_probability/calibrator_sc_v1.pkl",
            "features":  CFG.sc_features,
            "threshold": CFG.sc_threshold,
            "target":    "sc_within_3_laps",
            "source_notebook": "N14_sc_model",
        },
        "threat_levels": {
            "high_overtake":   CFG.high_overtake,
            "medium_overtake": CFG.medium_overtake,
            "high_sc":         CFG.high_sc,
            "medium_sc":       CFG.medium_sc,
        },
        "tools": ["predict_overtake_tool", "predict_sc_tool"],
        "llm": "ChatOpenAI — openai / lmstudio",
        "entry_point": "run_race_situation_agent",
        "input_schema": {
            "session":     "FastF1 Session object (loaded with laps + weather)",
            "driver":      "str — FastF1 driver abbreviation",
            "rival_ahead": "str | None — car directly ahead (None if gap > 2.5s)",
            "lap_number":  "int — current lap",
            "gp_name":     "str — GP short name for cluster map lookup",
            "event_name":  "str — full event name for SC rate map lookup",
            "year":        "int — race year",
        },
        "output_schema": {
            "overtake_prob": "float — calibrated P(overtake within ~3 laps)",
            "sc_prob_3lap":  "float — calibrated P(SC deployment within 3 laps)",
            "threat_level":  "str — LOW / MEDIUM / HIGH",
            "gap_ahead_s":   "float — on-track gap to car ahead (s)",
            "pace_delta_s":  "float — lap time delta vs rival (s, positive = slower)",
            "reasoning":     "str — LLM-generated threat assessment",
        },
    }

    out_path = CFG.export_dir / "race_situation_agent_config_v1.json"
    out_path.write_text(json.dumps(cfg_dict, indent=2), encoding="utf-8")
    print(f"Config exported → {out_path.relative_to(repo_root)}")
    print(json.dumps(cfg_dict, indent=2))
    return out_path

export_race_situation_agent_config()

Config exported → data\models\agents\race_situation_agent_config_v1.json
{
  "agent": "N27_race_situation_agent",
  "version": "v1",
  "overtake_model": {
    "file": "data/models/overtake_probability/lgbm_overtake_v1.pkl",
    "calibrator": "data/models/overtake_probability/calibrator.pkl",
    "features": [
      "gap_ahead_s",
      "pace_delta_s",
      "tyre_life_x",
      "tyre_life_y",
      "tyre_life_diff",
      "speed_trap_delta",
      "LapNumber",
      "drs_window",
      "compound_x",
      "compound_y",
      "circuit_cluster",
      "gap_pace_product",
      "drs_ready_gap",
      "gap_trend",
      "pace_delta_rolling3"
    ],
    "cat_features": [
      "compound_x",
      "compound_y",
      "circuit_cluster"
    ],
    "threshold": 0.7976,
    "source_notebook": "N12_overtake_model"
  },
  "sc_model": {
    "file": "data/models/safety_car_probability/lgbm_sc_v1.pkl",
    "calibrator": "data/models/safety_car_probability/calibrator_sc_v1.pkl",
    "features": [
    

WindowsPath('c:/Users/victo/Desktop/Documents/Cuarto Año/TFG/F1_Strat_Manager/data/models/agents/race_situation_agent_config_v1.json')

### Step 6 — Results

Config exported to `data/models/agents/race_situation_agent_config_v1.json`.

```
Config exported → data\models\agents\race_situation_agent_config_v1.json
```

The JSON captures the full agent schema: both model file paths, feature lists,
thresholds, threat-level boundaries, and source notebook references — everything
N31 needs to reconstruct or audit the agent's decision logic without re-running
this notebook.
